# Bayesian spatial models: a walkthrough

By the end of this lesson you will have fitted six models to one dataset,
followed the package's specification tests from OLS to the model they point to,
and used Bayes factors to rank the models those tests leave standing.

You do not need to know the spatial econometrics beforehand — each model is
introduced when the previous one runs out of road. Work through the cells in
order; every step produces something you can look at.

Every model here accepts a formula and a weights graph:

```python
SAR(formula="y ~ x1 + x2", data=gdf, W=W)
```

The equations behind each one are in [Supported Models](../models.md) if you
want them; you will not need them to follow along.

In [ ]:
import arviz as az
import geodatasets
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from libpysal.graph import Graph

from neighbayes.diagnostics.bayesfactor import bayes_factor_compare_models
from neighbayes.models import OLS, SAR, SDEM, SDM, SEM, SLX

az.style.use("arviz-variat")

In [ ]:
# Load sample spatial dataset
gdf = gpd.read_file(geodatasets.data.geoda.airbnb.url)
xcols = ["poverty", "rev_rating", "num_spots", "crowded"]
ycol = "price_pp"
gdf = gdf.dropna(subset=xcols + [ycol]).copy()

# Build contiguity graph (queen)
W = Graph.build_contiguity(gdf, rook=False).transform("r")

print(f"Observations: {len(gdf)}")
print(f"Predictors: {xcols}")

__Helper Functions__

To keep each model section focused, we use utility functions to:

* fit with consistent MCMC settings,
* print posterior summaries,
* tabulate direct/indirect/total effects.

For a quick pedagogical run, we use small draw counts. Increase these for real inference.


In [ ]:
def fit_and_report(model_cls, formula, data, W, draws=400, tune=400, chains=4, seed=42):
    """Fit a spatial model and return (model, summary, effects_df).

    Uses minimal MCMC settings for pedagogical demonstration.
    Increase draws/tune for real analyses.
    """
    model = model_cls(formula=formula, data=data, W=W, logdet_method="eigenvalue")
    model.fit(
        draws=draws,
        tune=tune,
        chains=chains,
        random_seed=seed,
        progressbar=False,
        idata_kwargs={"log_likelihood": True},
    )
    summary = model.summary(round_to=3)
    effects_df = pd.DataFrame(model.impacts())
    return model, summary, effects_df

__Convergence Diagnostics Helper__

For each fitted model, we will inspect:

* `r_hat` (target close to 1.00)
* effective sample sizes (`ess_bulk`, `ess_tail`)
* trace plots for key parameters.


In [ ]:
def diagnostics_table(idata, var_names):
    """Show key MCMC diagnostics for the given parameters."""
    cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
    return az.summary(idata, var_names=var_names, round_to=3)[cols]


def show_trace(idata, var_names, title):
    """Plot trace plots for the given parameters."""
    az.plot_trace_dist(idata, var_names=var_names)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

## The models, one at a time


### Start with OLS, and see what it misses


Model:

$$
y = X\beta + \epsilon
$$


In [ ]:
ols = OLS(
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
olsfit = ols.fit(
    draws=400,
    tune=400,
    chains=4,
    random_seed=42,
    progressbar=False,
    idata_kwargs={"log_likelihood": True},
)

In [ ]:
ols.summary()

### Add the neighbors' covariates (SLX)

Model:

$$
y = X\beta + WX\theta + \epsilon
$$

Interpretation:

* `beta` captures local covariate effects.
* `theta` captures neighbor-covariate spillovers.
* No spatial lag on $y$, so no autoregressive propagation through outcomes.


In [ ]:
slx, summary_slx, effects_slx = fit_and_report(
    SLX,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_slx)
display(effects_slx)

In [ ]:
az.plot_forest(slx.inference_data)
display(diagnostics_table(slx.inference_data, ["beta", "sigma"]))
show_trace(slx.inference_data, ["sigma"], "SLX Trace: sigma")

### Let the outcome itself spill over (SAR)

Model:

$$
y = \rho Wy + X\beta + \epsilon
$$

Interpretation:

* $\rho$ controls feedback from neighboring outcomes.
* Effects are amplified through $(I - \rho W)^{-1}$, so direct and indirect impacts differ from raw $\beta$.


In [ ]:
sar, summary_sar, effects_sar = fit_and_report(
    SAR,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
summary_sar

In [ ]:
effects_sar

In [ ]:
az.plot_forest(sar.inference_data)
display(diagnostics_table(sar.inference_data, ["rho", "beta", "sigma"]))
show_trace(sar.inference_data, ["rho", "sigma"], "SAR Trace: rho, sigma")

### Treat the clustering as nuisance instead (SEM)

Model:

$$
y = X\beta + u, \quad u = \lambda Wu + \epsilon
$$

Interpretation:

* Spatial dependence is moved to latent shocks rather than outcome feedback.
* Useful when omitted spatial factors induce correlated residuals.


In [ ]:
sem, summary_sem, effects_sem = fit_and_report(
    SEM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sem)
display(effects_sem)

In [ ]:
az.plot_forest(sem.inference_data)
display(diagnostics_table(sem.inference_data, ["lam", "beta", "sigma"]))
show_trace(sem.inference_data, ["lam", "sigma"], "SEM Trace: lam, sigma")

### Combine both channels (SDM)

Model:

$$
y = \rho Wy + X\beta + WX\theta + \epsilon
$$

Interpretation:

* Includes both outcome feedback and neighbor-covariate channels.
* Often used as a flexible nesting model for SAR/SLX.


In [ ]:
sdm, summary_sdm, effects_sdm = fit_and_report(
    SDM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sdm)
display(effects_sdm)

In [ ]:
az.plot_forest(sdm.inference_data)
display(diagnostics_table(sdm.inference_data, ["rho", "beta", "sigma"]))
show_trace(sdm.inference_data, ["rho", "sigma"], "SDM Trace: rho, sigma")

### The error-side combination (SDEM)

Model:

$$
y = X\beta + WX\theta + u, \quad u = \lambda Wu + \epsilon
$$

Interpretation:

* Neighbor covariates matter directly (through $WX$),
* and unmodeled shocks are spatially correlated (through $u$).


In [ ]:
sdem, summary_sdem, effects_sdem = fit_and_report(
    SDEM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sdem)
display(effects_sdem)

In [ ]:
az.plot_forest(sdem.inference_data)
display(diagnostics_table(sdem.inference_data, ["lam", "beta", "sigma"]))
show_trace(sdem.inference_data, ["lam", "sigma"], "SDEM Trace: lam, sigma")

### Check the chains before comparing anything

Bayesian estimation of spatial models has a well-known efficiency
pitfall: the spatial dependence parameter $\rho$ (or $\lambda$) often
mixes slowly, so a chain that _looks_ converged in its point estimate
can still produce posterior credible intervals that are 10–12 % too
narrow because the sampler has not visited the tails enough times
{cite:p}`wolf2018StochasticEfficiency`. The
`spatial_mcmc_diagnostic` helper checks effective sample size, sampler
yield, $\hat{R}$, and HPDI stability for the spatial parameter and
warns when any threshold is violated.


In [ ]:
from neighbayes.diagnostics import spatial_mcmc_diagnostic

# Run on the SDM (most parameters; both rho and impact summaries depend on it)
report = spatial_mcmc_diagnostic(sdm, emit_warnings=False)
report.to_frame()

## Let the tests pick the specification

Every fitted model can test for the spatial terms it leaves out. The tests are
Lagrange-multiplier (score) tests, which need only the fit you already have:
`model.spatial_diagnostics()` runs the ones that apply to that model, and
`model.spatial_diagnostics_decision()` reads them and recommends a model.

A fitted model that leaves out two terms has a plain and a robust test for
each. SAR leaves out the neighbors' covariates $WX\theta$ and the error process
$\lambda Wu$; OLS leaves out both and the lag $\rho Wy$ as well. A plain
test is valid when the other term is absent. A robust test stays valid when the
other term is present, and pays for that in power. So the decision reads them in
order:

1. Neither plain test rejects: keep the model.
2. One rejects: add that term.
3. Both reject: add the term whose robust test rejects; if both or neither do,
   the one with the smaller robust p-value.

This is the specific-to-general strategy of Anselin et al. (1996), the one
`spreg`'s `stge_classic` implements. You fit the recommended model and ask
again, until the tests stop rejecting. The p-values here are Bayesian: each
statistic is evaluated at the posterior mean, and its posterior spread gives
the credible interval beside it.

In [ ]:
def read_tests(model, alpha=0.05):
    """Show a fitted model's LM tests and say which reject."""
    diag = model.spatial_diagnostics()
    rejecting = diag.index[diag["p_value"] < alpha].tolist()
    name = type(model).__name__
    print(f"{name}: tests rejecting at {alpha}: {', '.join(rejecting) or 'none'}")
    print(
        f"{name}: recommended model: "
        f"{model.spatial_diagnostics_decision(alpha=alpha, format='model')}"
    )
    return diag[["statistic", "df", "p_value", "ci_lower", "ci_upper"]].round(3)

### Start from OLS

The OLS tests ask whether the outcome, the errors or the neighbors' covariates
carry spatial dependence. A lone rejection points to the SAR, SEM or SLX. When
the lag and the error both reject, their robust pair decides, and the $WX$
question waits for the SAR or SEM fit. When the lag and $WX$ both reject,
Koley and Bera's robust pair decides: $WX$ robust to the lag (Robust-LM-WX), and
the lag robust to $WX$, which at the OLS fit is the error test. The lag's score
there is the error's score plus a fixed combination of the $WX$ scores, which is
also why the two joint tests reported beside the tree are one statistic.

In [ ]:
read_tests(ols)

In [ ]:
ols.spatial_diagnostics_decision()

### Refit, and ask again

The OLS tree names the model to fit next, and that model tests the terms it
still leaves out:

- the SAR, the neighbors' covariates (towards the SDM) and the error process
  (towards SARAR, a SAR with spatial errors, which neighbayes does not fit);
- the SLX, the lag (towards the SDM) and the error process (towards the SDEM);
- the SEM, the lag (towards SARAR) and the neighbors' covariates (towards the
  SDEM);
- the SDM and SDEM, one term each, whose test decides whether to go on to the
  general nesting model (GNS).

The walk ends at the first model whose tests do not reject. The cell below
follows the recommendations through the models fitted above and draws each tree
it consults.

In [ ]:
def walk(start, fitted, alpha=0.05, show=True):
    """Follow the recommendations from ``start`` through the fitted models."""
    model, path = start, [type(start).__name__]
    while True:
        nxt = model.spatial_diagnostics_decision(alpha=alpha, format="model")
        if nxt == path[-1]:
            break
        if nxt not in fitted:
            path.append(f"{nxt} (not fitted here)")
            break
        model = fitted[nxt]
        path.append(nxt)
        if show:
            print(f"--- at the {nxt} fit ---")
            display(read_tests(model, alpha))
            display(model.spatial_diagnostics_decision(alpha=alpha))
    if show:
        print(" → ".join(path))
    return path


fitted = {"SLX": slx, "SAR": sar, "SEM": sem, "SDM": sdm, "SDEM": sdem}
path = walk(ols, fitted)

## Compare what you have fitted


In [ ]:
# Collect all fitted models for comparison
model_dict = {
    "OLS": ols,
    "SLX": slx,
    "SAR": sar,
    "SEM": sem,
    "SDM": sdm,
    "SDEM": sdem,
}
idata_dict = {name: m.inference_data for name, m in model_dict.items()}

### Bayes factors

The LM tests are misspecification tests: each asks whether a fitted model has
left spatial dependence behind. A Bayes factor asks something else, how well a
model fits overall. It compares marginal likelihoods, the probability of the
data under each model averaged over its prior:

$$BF_{ij} = \frac{p(y \mid \mathcal{M}_i)}{p(y \mid \mathcal{M}_j)}.$$

Because the average is over the prior, a coefficient the data do not need costs
its model: the prior spreads probability over values the data rule out. This is
the Bayesian Occam's razor.

For these six Gaussian models the marginal likelihood reduces to a
two-dimensional integral, so `method="quadrature"` computes it exactly, with no
Monte Carlo error and no fit. Bridge sampling (`method="bridge"`) estimates the
same quantity from the posterior draws, and is what models without that form
use; BIC approximates it as if each prior held one observation's worth of
information. The table gives each model's log marginal likelihood relative to
the best, so 0 marks the best model and $-\log 10 \approx -2.3$ a Bayes factor
of 10 against.

| Bayes factor | Evidence (Kass & Raftery, 1995) |
| ------------ | ------------------------------- |
| 1 – 3        | Anecdotal                       |
| 3 – 10       | Moderate                        |
| 10 – 30      | Strong                          |
| 30 – 100     | Very strong                     |
| > 100        | Extreme                         |

In [ ]:
from neighbayes.diagnostics import post_prob

log_bf = {
    method: bayes_factor_compare_models(model_dict, method=method, log=True)
    for method in ("quadrature", "bridge", "bic")
}
exact = log_bf["quadrature"]
best = exact.sum(axis=1).idxmax()  # the row that beats every column
comparison = pd.DataFrame({m: bf[best] for m, bf in log_bf.items()})
comparison["posterior probability"] = post_prob(
    comparison["quadrature"].to_numpy(), model_names=list(comparison.index)
)
comparison.sort_values("quadrature", ascending=False).round(3)

`bayes_factor_compare_models` returns the full matrix of pairwise log Bayes
factors. Each entry is the row model over the column model, so a positive
entry favors the row; the column above is the matrix's column for the best
model.

In [ ]:
exact.round(2)

### Rank only the models that pass their tests

Because the two measure different things, they need not agree. A model can fit
well overall and still leave residual spatial dependence (in the outcome, the
errors or the neighbors' covariates), and its estimates are then biased or
inefficient. So a model that fails an LM test cannot be trusted however its
Bayes factor ranks it. Use the tests to screen the models and the Bayes factors
to rank the ones that pass.

The cell below lists each fitted model's rejecting tests beside its log marginal
likelihood, and picks the best model among those that pass.

In [ ]:
alpha = 0.05


def rejecting(model):
    diag = model.spatial_diagnostics()
    return ", ".join(diag.index[diag["p_value"] < alpha]) or "none"


screen = pd.DataFrame(
    {
        "rejecting tests": {name: rejecting(m) for name, m in model_dict.items()},
        "log ML vs best": exact[best],
    }
)
passing = screen.index[screen["rejecting tests"] == "none"]
print(f"Best by Bayes factor alone: {best}")
print(
    f"Best among the models that pass their tests: "
    f"{screen.loc[passing, 'log ML vs best'].idxmax() if len(passing) else 'none'}"
)
print(f"End of the walk: {path[-1]}")
screen.sort_values("log ML vs best", ascending=False).round(3)

## See how the spatial parameter moves across models

This cell compares posterior means/intervals for the spatial scalar where present:

* `rho` in SAR/SDM
* `lam` in SEM/SDEM


In [ ]:
# Compare spatial parameters across models
spatial_rows = []
for name, model, var in [
    ("SAR", sar, "rho"),
    ("SEM", sem, "lam"),
    ("SDM", sdm, "rho"),
    ("SDEM", sdem, "lam"),
]:
    if var in model.inference_data.posterior:
        summary = az.summary(model.inference_data, var_names=[var], round_to=3)
        summary.insert(0, "model", name)
        spatial_rows.append(summary)

pd.concat(spatial_rows)

## Where to go next

- [How to run Bayesian LM specification tests](../how-to/bayesian_lmtests.ipynb)
  — the tests used above, in full
- [How to run spatial block cross-validation](../how-to/spatial_cv_demo.ipynb)
  — an out-of-sample check on the specification you chose
- [How to set priors](../how-to/priors.ipynb)
- [Supported Models](../models.md) — every model and its equations